In [3]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

def get_us_tickers():
    candidate_urls = [
        "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt",
        "https://ftp.nasdaqtrader.com/SymDir/nasdaqtraded.txt",
    ]

    last_err = None
    for url in candidate_urls:
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            df = pd.read_csv(StringIO(r.text), sep="|")
            break
        except Exception as e:
            last_err = e
    else:
        raise RuntimeError(f"Could not download ticker universe. Last error: {last_err}")

    df = df[df["Test Issue"] == "N"].copy()
    df = df[df["ETF"] != "Y"].copy()   # remove ETFs if you want only equities
    df = df[df["Symbol"].notna()].copy()

    tickers = df["Symbol"].astype(str).tolist()

    # yfinance compatibility
    bad_suffixes = ["W", "R", "P", "Q"]
    cleaned = []
    for t in tickers:
        t = t.strip()
        if not t or "$" in t:
            continue
        t = t.replace(".", "-")
        # optionally skip weird preferred/unit/right tickers
        if len(t) > 4 and t[-1] in bad_suffixes:
            continue
        cleaned.append(t)

    cleaned = sorted(set(cleaned))
    return cleaned

tickers = get_us_tickers()
print(len(tickers), tickers[:20])

6423 ['A', 'AA', 'AACB', 'AACBU', 'AACG', 'AACIU', 'AACOU', 'AAL', 'AAME', 'AAMI', 'AAOI', 'AAON', 'AAP', 'AAPG', 'AAPL', 'AARD', 'AAT', 'AAUC', 'AB', 'ABAT']


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import time

START = "2023-01-01"
MARKET = "SPY"

MIN_PRICE = 5
MIN_OBS = 120
MIN_MKT_CAP = 1e8
TOP_N = 100

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def download_close_batch(tickers, start):
    try:
        px = yf.download(
            tickers,
            start=start,
            auto_adjust=True,
            progress=False,
            threads=True,
            group_by="ticker",
        )

        if px is None or px.empty:
            return pd.DataFrame()

        # Handle both possible MultiIndex layouts from yfinance:
        # 1) (ticker, field) and 2) (field, ticker)
        if isinstance(px.columns, pd.MultiIndex):
            lv0 = px.columns.get_level_values(0)
            lv1 = px.columns.get_level_values(1)

            if "Close" in lv1:
                close = px.xs("Close", axis=1, level=1)
            elif "Adj Close" in lv1:
                close = px.xs("Adj Close", axis=1, level=1)
            elif "Close" in lv0:
                close = px.xs("Close", axis=1, level=0)
            elif "Adj Close" in lv0:
                close = px.xs("Adj Close", axis=1, level=0)
            else:
                return pd.DataFrame()

            # If only one symbol comes back, force DataFrame
            if isinstance(close, pd.Series):
                close = close.to_frame(name=tickers[0])
            return close

        # Single-symbol non-MultiIndex case
        for field in ["Close", "Adj Close"]:
            if field in px.columns:
                close = px[[field]].copy()
                close.columns = [tickers[0]]
                return close

        return pd.DataFrame()
    except Exception:
        return pd.DataFrame()

# 1. universe
tickers = get_us_tickers()

# optional first-pass restriction so this finishes
tickers = tickers[:3000]

# 2. download prices in batches
all_prices = []
for batch in chunked(tickers + [MARKET], 200):
    px = download_close_batch(batch, START)
    if not px.empty:
        all_prices.append(px)
    time.sleep(0.5)

if not all_prices:
    raise RuntimeError("No price data downloaded from Yahoo.")

prices = pd.concat(all_prices, axis=1)
prices = prices.loc[:, ~prices.columns.duplicated()]

# Make sure market proxy exists before filters
if MARKET not in prices.columns:
    # One direct retry for SPY
    spy_px = download_close_batch([MARKET], START)
    if not spy_px.empty and MARKET in spy_px.columns:
        prices = pd.concat([prices, spy_px[[MARKET]]], axis=1)
        prices = prices.loc[:, ~prices.columns.duplicated()]

if MARKET not in prices.columns:
    raise ValueError("SPY missing from prices after download.")

# 3. basic price filter
latest_price = prices.ffill().iloc[-1]
valid_cols = latest_price[latest_price >= MIN_PRICE].index.tolist()
if MARKET not in valid_cols:
    valid_cols.append(MARKET)
prices = prices[valid_cols]

# 4. returns
rets = np.log(prices / prices.shift(1))
rets = rets.dropna(how="all")

if MARKET not in rets.columns:
    raise ValueError("SPY missing from returns. Add SPY separately.")

mkt = rets[MARKET]
print(f"Price columns after filter: {prices.shape[1]}")
print(f"Return columns: {rets.shape[1]}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ALUB-U"}}}
$ALUB-U: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AIIA-U"}}}
$AIIA-U: possibly delisted; no timezone found

2 Failed downloads:
['ALUB-U', 'AIIA-U']: possibly delisted; no timezone found
$BEBE-U: possibly delisted; no timezone found
$BIII-U: possibly delisted; no timezone found
$BCSS-U: possibly delisted; no timezone found

3 Failed downloads:
['BEBE-U', 'BIII-U', 'BCSS-U']: possibly delisted; no timezone found
$BUI-V: possibly delisted; no timezone found

1 Failed download:
['BUI-V']: possibly delisted; no timezone found
$COPL-U: possibly delisted; no timezone found
$CLBR-U: possibly delisted; no timezone found

2 Failed downloads:
['COPL-U', 'CLBR-U']: possibly delisted; no timezone found
$EVAC-U: possibly delisted; no timezone found
$FCRS-U: po

Empty DataFrame
Columns: [ticker, beta, r2, nobs, vol, market_cap, last_price]
Index: []


In [7]:
# 5. beta estimates
rows = []
for col in rets.columns:
    if col == MARKET:
        continue

    df = pd.concat([rets[col], mkt], axis=1).dropna()
    if len(df) < MIN_OBS:
        continue

    X = sm.add_constant(df[MARKET])
    y = df[col]
    try:
        model = sm.OLS(y, X).fit()
        rows.append({
            "ticker": col,
            "beta": model.params[MARKET],
            "r2": model.rsquared,
            "nobs": len(df),
            "vol": y.std() * np.sqrt(252),
        })
    except Exception:
        pass

betas = pd.DataFrame(rows).sort_values("beta", ascending=False)
print(f"Beta rows computed: {len(betas)}")

if betas.empty:
    raise RuntimeError(
        "No beta rows were computed. Check close-price extraction and MIN_OBS threshold."
    )

# 6. metadata pass only for top names
meta_rows = []
for t in betas["ticker"].head(500):
    market_cap = np.nan
    last_price = np.nan
    try:
        tk = yf.Ticker(t)

        # Fast path
        try:
            fi = tk.fast_info
        except Exception:
            fi = {}

        if hasattr(fi, "get"):
            market_cap = fi.get("market_cap", fi.get("marketCap", np.nan))
            last_price = fi.get("last_price", fi.get("lastPrice", np.nan))

        # Fallback path
        if pd.isna(market_cap) or pd.isna(last_price):
            try:
                inf = tk.info or {}
            except Exception:
                inf = {}

            if pd.isna(market_cap):
                market_cap = inf.get("marketCap", np.nan)
            if pd.isna(last_price):
                last_price = inf.get(
                    "currentPrice",
                    inf.get("regularMarketPrice", np.nan),
                )

    except Exception:
        pass

    meta_rows.append({
        "ticker": t,
        "market_cap": market_cap,
        "last_price": last_price,
    })

meta = pd.DataFrame(meta_rows)

out = betas.merge(meta, on="ticker", how="left")

# Apply market-cap filter, but avoid returning empty output if metadata lookup failed
filtered = out[out["market_cap"].fillna(0) >= MIN_MKT_CAP]
if len(filtered) == 0 and len(out) > 0:
    print(
        "Warning: market_cap lookup returned mostly NaN/0. Showing unfiltered beta table. "
        "(Try rerun later or lower MIN_MKT_CAP.)"
    )
else:
    out = filtered

out = out.sort_values(["beta", "market_cap"], ascending=[False, False])
print(out.head(TOP_N))

Beta rows computed: 2069
    ticker      beta        r2  nobs       vol    market_cap  last_price
0      DEC       inf       NaN   572       NaN  1.120252e+09   14.610000
1     BMNR  6.012044  0.053011   193  2.912248  9.342875e+09   20.540001
2     GLXY  3.973846  0.246877   206  0.904027  8.730141e+09   22.350000
4     FIGR  3.647093  0.139239   126  1.159511  7.568441e+09   35.070000
7     AAOI  3.369923  0.165569   800  1.251323  7.279997e+09   96.809998
..     ...       ...       ...   ...       ...           ...         ...
138   ACDC  1.686220  0.099410   800  0.808050  1.148532e+09    6.350000
140    GFS  1.684157  0.335540   800  0.439288  2.326949e+10   41.860001
141   HIMX  1.682724  0.198371   800  0.570838  1.721148e+09    9.840000
142    HRI  1.679816  0.240211   800  0.517850  3.634355e+09  108.910004
143   INTC  1.669575  0.213048   800  0.546520  2.286293e+11   45.770000

[100 rows x 7 columns]


In [5]:
mkt

Date
2023-01-04    0.007691
2023-01-05   -0.011479
2023-01-06    0.022673
2023-01-09   -0.000567
2023-01-10    0.006988
                ...   
2026-03-09    0.008722
2026-03-10   -0.001608
2026-03-11   -0.001256
2026-03-12   -0.015301
2026-03-13   -0.005676
Name: SPY, Length: 800, dtype: float64